# Email Subject Matching - Robust Optimization Notebook

This notebook does all steps end-to-end:
1. Auto-detect dataset paths
2. Build robust validation folds
3. Search model parameters with hard-case emphasis
4. Re-check top candidates on holdout folds
5. Train on full data and export submission

In [1]:
import os
from pathlib import Path

REQUIRED = {"train.csv", "test.csv", "test_subjects.csv"}
OPTIONAL = {"sample_submission.csv"}

search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("../input"),
    Path("./dataset/public"),
    Path("."),
]

def find_dataset_dirs(roots):
    found = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for dirpath, _, filenames in os.walk(root):
            files = set(filenames)
            if REQUIRED.issubset(files):
                p = Path(dirpath).resolve()
                key = str(p)
                if key not in seen:
                    seen.add(key)
                    found.append(p)
    return found

candidates = find_dataset_dirs(search_roots)
if not candidates:
    raise FileNotFoundError("No dataset folder found with train.csv, test.csv, test_subjects.csv")

def rank_path(p):
    files = set(os.listdir(p))
    has_optional = 0 if OPTIONAL.issubset(files) else 1
    short_depth = len(p.parts)
    return (has_optional, short_depth)

DATA_DIR = sorted(candidates, key=rank_path)[0]
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TEST_SUBJECTS_PATH = DATA_DIR / "test_subjects.csv"

if Path("/kaggle/working").exists():
    WORK_DIR = Path("/kaggle/working")
else:
    WORK_DIR = Path("./working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset candidates:")
for p in candidates:
    print(" -", p)
print("\nSelected DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)

Dataset candidates:
 - /kaggle/input/datasets/soufianelasfar/dataset

Selected DATA_DIR: /kaggle/input/datasets/soufianelasfar/dataset
WORK_DIR: /kaggle/working


In [2]:
import json
import random
import re
import subprocess
import sys
from dataclasses import dataclass
from itertools import permutations

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    try:
        __import__(import_name)
    except ImportError:
        pkg = pip_name or import_name
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("sentence_transformers", "sentence-transformers>=3.0.0")
from sentence_transformers import CrossEncoder, SentenceTransformer

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not enabled. Switch Kaggle accelerator to GPU (preferably P100) and rerun.")

DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name(0)
print("CUDA device:", GPU_NAME)
if "P100" not in GPU_NAME.upper():
    print("Warning: non-P100 GPU detected. Pipeline still runs, but runtime may differ.")

TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
ENTITY_RE = re.compile(r"\b(Project|Program|Initiative|Stream)\s+[A-Z][A-Za-z0-9-]*\b")

STOPWORDS = {
    "the",
    "a",
    "an",
    "and",
    "or",
    "to",
    "for",
    "of",
    "on",
    "in",
    "at",
    "with",
    "from",
    "by",
    "is",
    "are",
    "be",
    "this",
    "that",
    "it",
    "as",
    "we",
    "you",
    "your",
    "our",
    "please",
    "regards",
    "thanks",
    "hi",
    "hello",
    "re",
    "fw",
}

PERMS = list(permutations(range(4)))

BI_MODEL_SPECS = {
    "mini": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "body_mode": "plain",
        "subject_mode": "plain",
    },
    "e5_subj_query": {
        "name": "intfloat/e5-base-v2",
        "body_mode": "passage",
        "subject_mode": "query",
    },
    "e5_body_query": {
        "name": "intfloat/e5-base-v2",
        "body_mode": "query",
        "subject_mode": "passage",
    },
}

CROSS_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"


def apply_mode(text: str, mode: str) -> str:
    if mode == "query":
        return f"query: {text}"
    if mode == "passage":
        return f"passage: {text}"
    return text


def zscore_matrix(mat: np.ndarray) -> np.ndarray:
    m = mat.astype(np.float32)
    mu = float(m.mean())
    sigma = float(m.std())
    if sigma < 1e-6:
        return m - mu
    return (m - mu) / sigma


@dataclass
class BlockCache:
    category: str
    body_texts: list
    subject_texts: list
    body_tokens: list
    subj_tokens: list
    body_entities: list
    subj_entities: list
    tfidf_z: np.ndarray
    bi_z: dict
    ce_z: dict
    gt: np.ndarray


def token_set(text: str) -> set:
    return {
        t.lower()
        for t in TOKEN_RE.findall(str(text))
        if len(t) >= 3 and t.lower() not in STOPWORDS
    }


def entity_set(text: str) -> set:
    return {m.group(0) for m in ENTITY_RE.finditer(str(text))}


def split_train_val(train_df: pd.DataFrame, seed: int, train_frac: float = 0.8) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    tr_idx, va_idx = [], []
    for _, g in train_df.groupby("category", sort=False):
        idx = g.index.to_numpy().copy()
        rng.shuffle(idx)
        cut = int(len(idx) * train_frac)
        tr_idx.extend(idx[:cut])
        va_idx.extend(idx[cut:])
    tr = train_df.loc[tr_idx].reset_index(drop=True)
    va = train_df.loc[va_idx].reset_index(drop=True)
    return tr, va


def make_val_blocks(df: pd.DataFrame, seed: int) -> list:
    rng = np.random.default_rng(seed)
    blocks = []
    for cat, g in df.groupby("category", sort=False):
        idx = g.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx) // 4
        for b in range(n):
            sel = idx[4 * b : 4 * b + 4]
            rows = g.loc[sel].reset_index(drop=True)
            bodies = rows["body"].astype(str).tolist()
            true_subjects = rows["subject"].astype(str).tolist()
            perm = rng.permutation(4)
            subjects = [true_subjects[p] for p in perm]
            inv = np.empty(4, dtype=np.int64)
            for col, p in enumerate(perm):
                inv[p] = col
            blocks.append((cat, bodies, subjects, inv))
    return blocks


def fit_tfidf_vectorizers(train_df: pd.DataFrame) -> tuple[TfidfVectorizer, TfidfVectorizer]:
    corpus = pd.concat([train_df["body"], train_df["subject"]], axis=0).astype(str).tolist()
    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        norm="l2",
        max_features=120000,
    )
    char_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        norm="l2",
        max_features=180000,
    )
    word_vec.fit(corpus)
    char_vec.fit(corpus)
    return word_vec, char_vec


def tfidf_matrix(bodies: list[str], subjects: list[str], word_vec: TfidfVectorizer, char_vec: TfidfVectorizer) -> np.ndarray:
    bw = word_vec.transform(bodies)
    sw = word_vec.transform(subjects)
    bc = char_vec.transform(bodies)
    sc = char_vec.transform(subjects)
    word_sim = (bw @ sw.T).toarray()
    char_sim = (bc @ sc.T).toarray()
    return (0.6 * word_sim + 0.4 * char_sim).astype(np.float32)


def cosine_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return a @ b.T


def build_association_scores(
    train_df: pd.DataFrame,
    neg_per_pos: int,
    smooth_alpha: float,
    min_token_freq: int = 8,
    min_log_ratio: float = 0.02,
) -> dict:
    rng = np.random.default_rng(RANDOM_SEED)
    pos_counts = {}
    neg_counts = {}
    body_freq = {}
    subj_freq = {}

    grouped = {cat: g.reset_index(drop=True) for cat, g in train_df.groupby("category", sort=False)}

    def add_count(counter: dict, key, amount: int = 1) -> None:
        counter[key] = counter.get(key, 0) + amount

    for _, row in train_df.iterrows():
        bt = list(token_set(row["body"]))
        st = list(token_set(row["subject"]))

        for x in bt:
            add_count(body_freq, x)
        for x in st:
            add_count(subj_freq, x)

        for b in bt:
            for s in st:
                add_count(pos_counts, (b, s))

    for _, g in grouped.items():
        n = len(g)
        for i in range(n):
            bt = list(token_set(g.loc[i, "body"]))
            for _ in range(neg_per_pos):
                j = int(rng.integers(0, n - 1))
                if j >= i:
                    j += 1
                st = list(token_set(g.loc[j, "subject"]))
                for b in bt:
                    for s in st:
                        add_count(neg_counts, (b, s))

    keep_b = {t for t, c in body_freq.items() if c >= min_token_freq}
    keep_s = {t for t, c in subj_freq.items() if c >= min_token_freq}

    scores = {}
    for (b, s), p in pos_counts.items():
        if b not in keep_b or s not in keep_s:
            continue
        n = neg_counts.get((b, s), 0)
        v = np.log((p + smooth_alpha) / (n + smooth_alpha))
        if v > min_log_ratio:
            scores[(b, s)] = float(v)

    return scores


def association_matrix(
    body_tokens: list,
    subj_tokens: list,
    body_entities: list,
    subj_entities: list,
    assoc_scores: dict,
    entity_w: float,
    assoc_w: float,
    overlap_w: float,
    jaccard_w: float,
) -> np.ndarray:
    out = np.zeros((4, 4), dtype=np.float32)
    for i in range(4):
        bt = body_tokens[i]
        be = body_entities[i]
        for j in range(4):
            st = subj_tokens[j]
            se = subj_entities[j]

            assoc_sum = 0.0
            for b in bt:
                best = 0.0
                for s in st:
                    v = assoc_scores.get((b, s), 0.0)
                    if v > best:
                        best = v
                assoc_sum += best

            overlap = len(bt & st)
            union = len(bt | st)
            jaccard = (overlap / union) if union else 0.0
            ent_overlap = len(be & se)

            out[i, j] = (
                entity_w * ent_overlap
                + assoc_w * assoc_sum
                + overlap_w * overlap
                + jaccard_w * jaccard
            )
    return out


def best_assignment(score_matrix: np.ndarray) -> tuple[np.ndarray, float]:
    best_perm = None
    best_score = -1e18
    second_best = -1e18

    for perm in PERMS:
        s = (
            score_matrix[0, perm[0]]
            + score_matrix[1, perm[1]]
            + score_matrix[2, perm[2]]
            + score_matrix[3, perm[3]]
        )
        if s > best_score:
            second_best = best_score
            best_score = s
            best_perm = perm
        elif s > second_best:
            second_best = s

    margin = float(best_score - second_best)
    return np.asarray(best_perm, dtype=np.int64), margin


def load_models() -> tuple[dict, CrossEncoder]:
    loaded_bi = {}
    for spec in BI_MODEL_SPECS.values():
        name = spec["name"]
        if name not in loaded_bi:
            print("Loading bi-encoder:", name)
            loaded_bi[name] = SentenceTransformer(name, device=DEVICE)

    bi_models = {key: loaded_bi[spec["name"]] for key, spec in BI_MODEL_SPECS.items()}

    print("Loading cross-encoder:", CROSS_MODEL_NAME)
    cross_model = CrossEncoder(CROSS_MODEL_NAME, device=DEVICE)
    print("Models loaded successfully.")

    return bi_models, cross_model


try:
    BI_MODELS, CROSS_MODEL = load_models()
except Exception as exc:
    raise RuntimeError(
        "Failed to load transformer models. Enable Kaggle Internet and keep GPU enabled, then rerun."
    ) from exc


def predict_cross_scores(pairs: list[tuple[str, str]], batch_size: int = 256, chunk_size: int = 4096) -> np.ndarray:
    out = []
    for start in range(0, len(pairs), chunk_size):
        part = pairs[start : start + chunk_size]
        scores = CROSS_MODEL.predict(part, batch_size=batch_size, show_progress_bar=False)
        out.append(np.asarray(scores, dtype=np.float32))
    if not out:
        return np.zeros((0,), dtype=np.float32)
    return np.concatenate(out, axis=0)


CUDA device: Tesla P100-PCIE-16GB
Loading bi-encoder: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading bi-encoder: intfloat/e5-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully.


In [3]:
import os
import platform
import torch

print('python:', platform.python_version())
print('torch:', torch.__version__)
print('torch cuda build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))
    print('gpu capability:', torch.cuda.get_device_capability(0))
    x = torch.randn(1024, 1024, device='cuda')
    y = torch.randn(1024, 1024, device='cuda')
    z = x @ y
    print('cuda matmul ok, mean:', float(z.mean().cpu()))
else:
    print('no gpu visible')

print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))

python: 3.12.12
torch: 2.3.1+cu121
torch cuda build: 12.1
cuda available: True
gpu name: Tesla P100-PCIE-16GB
gpu capability: (6, 0)
cuda matmul ok, mean: 0.010002652183175087
CUDA_VISIBLE_DEVICES: None


In [6]:
import subprocess
import sys

import torch

print('current torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))

need_fix = (
    torch.cuda.is_available()
    and torch.cuda.get_device_capability(0)[0] <= 6
    and '+cu128' in torch.__version__
)

if need_fix:
    print('Installing P100-compatible torch stack (cu121)...')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--index-url',
        'https://download.pytorch.org/whl/cu121',
        'torch==2.3.1',
        'torchvision==0.18.1',
        'torchaudio==2.3.1',
    ])
    print('Install complete. Restart kernel, then rerun from Cell 2.')
else:
    print('No torch compatibility install needed.')

current torch: 2.10.0+cu128
cuda build: 12.8
gpu: Tesla P100-PCIE-16GB capability: (6, 0)
Installing P100-compatible torch stack (cu121)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 50.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 116.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 35.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
test_subjects_df = pd.read_csv(TEST_SUBJECTS_PATH)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("test_subjects shape:", test_subjects_df.shape)

search_seeds = [13, 29]
holdout_seeds = [41, 77, 101]


def build_fold_cache(seed: int) -> dict:
    tr, va = split_train_val(train_df, seed=seed, train_frac=0.8)
    raw_blocks = make_val_blocks(va, seed=seed + 999)

    word_vec, char_vec = fit_tfidf_vectorizers(tr)

    unique_texts = sorted({txt for _, bodies, subjects, _ in raw_blocks for txt in (bodies + subjects)})

    bi_lookup = {}
    for key, spec in BI_MODEL_SPECS.items():
        model = BI_MODELS[key]

        body_inputs = [apply_mode(t, spec["body_mode"]) for t in unique_texts]
        subj_inputs = [apply_mode(t, spec["subject_mode"]) for t in unique_texts]

        body_emb = model.encode(
            body_inputs,
            batch_size=128,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        subj_emb = model.encode(
            subj_inputs,
            batch_size=128,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        bi_lookup[key] = {
            "body": {t: body_emb[i] for i, t in enumerate(unique_texts)},
            "subject": {t: subj_emb[i] for i, t in enumerate(unique_texts)},
        }

    sb_pairs = []
    bs_pairs = []
    meta = []
    for blk_idx, (_, bodies, subjects, _) in enumerate(raw_blocks):
        for i in range(4):
            for j in range(4):
                sb_pairs.append((subjects[j], bodies[i]))
                bs_pairs.append((bodies[i], subjects[j]))
                meta.append((blk_idx, i, j))

    sb_scores = predict_cross_scores(sb_pairs, batch_size=256, chunk_size=4096)
    bs_scores = predict_cross_scores(bs_pairs, batch_size=256, chunk_size=4096)

    ce_sb = [np.zeros((4, 4), dtype=np.float32) for _ in raw_blocks]
    ce_bs = [np.zeros((4, 4), dtype=np.float32) for _ in raw_blocks]

    for idx, (blk_idx, i, j) in enumerate(meta):
        ce_sb[blk_idx][i, j] = float(sb_scores[idx])
        ce_bs[blk_idx][i, j] = float(bs_scores[idx])

    block_caches = []
    for blk_idx, (cat, bodies, subjects, gt) in enumerate(raw_blocks):
        tfidf_z = zscore_matrix(tfidf_matrix(bodies, subjects, word_vec, char_vec))

        bi_z = {}
        for key in BI_MODEL_SPECS:
            b = np.vstack([bi_lookup[key]["body"][t] for t in bodies]).astype(np.float32)
            s = np.vstack([bi_lookup[key]["subject"][t] for t in subjects]).astype(np.float32)
            bi_z[key] = zscore_matrix(cosine_matrix(b, s))

        ce_z = {
            "none": np.zeros((4, 4), dtype=np.float32),
            "ce_subj_body": zscore_matrix(ce_sb[blk_idx]),
            "ce_body_subj": zscore_matrix(ce_bs[blk_idx]),
        }

        block_caches.append(
            BlockCache(
                category=cat,
                body_texts=bodies,
                subject_texts=subjects,
                body_tokens=[token_set(x) for x in bodies],
                subj_tokens=[token_set(x) for x in subjects],
                body_entities=[entity_set(x) for x in bodies],
                subj_entities=[entity_set(x) for x in subjects],
                tfidf_z=tfidf_z,
                bi_z=bi_z,
                ce_z=ce_z,
                gt=gt,
            )
        )

    return {
        "seed": seed,
        "train_df": tr,
        "blocks": block_caches,
    }


search_folds = [build_fold_cache(s) for s in search_seeds]
holdout_folds = [build_fold_cache(s) for s in holdout_seeds]

print("prepared search folds:", len(search_folds))
for f in search_folds:
    print(" seed", f["seed"], "train", f["train_df"].shape, "blocks", len(f["blocks"]))

print("prepared holdout folds:", len(holdout_folds))
for f in holdout_folds:
    print(" seed", f["seed"], "train", f["train_df"].shape, "blocks", len(f["blocks"]))

train shape: (3000, 4)
test shape: (820, 3)
test_subjects shape: (820, 3)
prepared search folds: 2
 seed 13 train (2400, 4) blocks 150
 seed 29 train (2400, 4) blocks 150
prepared holdout folds: 3
 seed 41 train (2400, 4) blocks 150
 seed 77 train (2400, 4) blocks 150
 seed 101 train (2400, 4) blocks 150


In [5]:
ASSOC_CACHE = {}


def get_assoc_scores_for_fold(fold: dict, neg_per_pos: int, smooth_alpha: float) -> dict:
    key = (int(fold["seed"]), int(neg_per_pos), float(smooth_alpha))
    if key not in ASSOC_CACHE:
        ASSOC_CACHE[key] = build_association_scores(
            fold["train_df"],
            neg_per_pos=int(neg_per_pos),
            smooth_alpha=float(smooth_alpha),
            min_token_freq=8,
            min_log_ratio=0.02,
        )
    return ASSOC_CACHE[key]


def evaluate_params_on_fold(fold: dict, params: dict) -> dict:
    assoc_scores = get_assoc_scores_for_fold(
        fold,
        neg_per_pos=params["neg_per_pos"],
        smooth_alpha=params["smooth_alpha"],
    )

    block_scores = []
    hard_scores = []

    for bc in fold["blocks"]:
        assoc_raw = association_matrix(
            bc.body_tokens,
            bc.subj_tokens,
            bc.body_entities,
            bc.subj_entities,
            assoc_scores=assoc_scores,
            entity_w=params["entity_w"],
            assoc_w=params["assoc_w"],
            overlap_w=params["overlap_w"],
            jaccard_w=params["jaccard_w"],
        )
        assoc_z = zscore_matrix(assoc_raw)

        final_mat = (
            params["w_assoc"] * assoc_z
            + params["w_tfidf"] * bc.tfidf_z
            + params["w_bi"] * bc.bi_z[params["bi_key"]]
            + params["w_ce"] * bc.ce_z[params["ce_key"]]
        )

        pred, margin = best_assignment(final_mat)
        acc = float(np.mean(pred == bc.gt))
        block_scores.append(acc)

        if margin <= 0.35:
            hard_scores.append(acc)

    overall = float(np.mean(block_scores)) if block_scores else np.nan
    hard = float(np.mean(hard_scores)) if hard_scores else overall
    objective = 0.30 * overall + 0.70 * hard

    return {
        "overall": overall,
        "hard": hard,
        "objective": objective,
    }


rng = np.random.default_rng(2026)

pool = {
    "neg_per_pos": [2, 3, 4],
    "smooth_alpha": [3.0, 4.0, 5.0],
    "entity_w": [1.2, 1.6, 2.0],
    "assoc_w": [0.7, 0.9, 1.1],
    "overlap_w": [0.25, 0.35, 0.45],
    "jaccard_w": [0.4, 0.6, 0.8],
    "bi_key": ["mini", "e5_subj_query", "e5_body_query"],
    "ce_key": ["none", "ce_subj_body", "ce_body_subj"],
    "w_assoc": [0.8, 1.0, 1.2, 1.4],
    "w_tfidf": [0.2, 0.4, 0.6, 0.8],
    "w_bi": [0.8, 1.2, 1.6, 2.0, 2.4],
    "w_ce": [0.0, 0.6, 1.0, 1.4, 1.8],
}


def sample_candidates(sample_n: int, rng: np.random.Generator) -> list:
    keys = [
        "neg_per_pos",
        "smooth_alpha",
        "entity_w",
        "assoc_w",
        "overlap_w",
        "jaccard_w",
        "bi_key",
        "ce_key",
        "w_assoc",
        "w_tfidf",
        "w_bi",
        "w_ce",
    ]

    out = []
    seen = set()

    baseline = {
        "neg_per_pos": 2,
        "smooth_alpha": 4.0,
        "entity_w": 1.6,
        "assoc_w": 0.8,
        "overlap_w": 0.35,
        "jaccard_w": 0.6,
        "bi_key": "mini",
        "ce_key": "none",
        "w_assoc": 1.0,
        "w_tfidf": 1.0,
        "w_bi": 0.0,
        "w_ce": 0.0,
    }
    out.append(baseline)
    seen.add(tuple(baseline[k] for k in keys))

    while len(out) < sample_n:
        cand = {
            "neg_per_pos": int(rng.choice(pool["neg_per_pos"])),
            "smooth_alpha": float(rng.choice(pool["smooth_alpha"])),
            "entity_w": float(rng.choice(pool["entity_w"])),
            "assoc_w": float(rng.choice(pool["assoc_w"])),
            "overlap_w": float(rng.choice(pool["overlap_w"])),
            "jaccard_w": float(rng.choice(pool["jaccard_w"])),
            "bi_key": str(rng.choice(pool["bi_key"])),
            "ce_key": str(rng.choice(pool["ce_key"])),
            "w_assoc": float(rng.choice(pool["w_assoc"])),
            "w_tfidf": float(rng.choice(pool["w_tfidf"])),
            "w_bi": float(rng.choice(pool["w_bi"])),
            "w_ce": float(rng.choice(pool["w_ce"])),
        }

        if cand["ce_key"] == "none":
            cand["w_ce"] = 0.0

        key = tuple(cand[k] for k in keys)
        if key in seen:
            continue

        seen.add(key)
        out.append(cand)

    return out


sample_n = 180
sampled_candidates = sample_candidates(sample_n=sample_n, rng=rng)

rows = []
for i, cand in enumerate(sampled_candidates, 1):
    fold_res = [evaluate_params_on_fold(fold, cand) for fold in search_folds]

    row = dict(cand)
    row["search_overall"] = float(np.mean([x["overall"] for x in fold_res]))
    row["search_hard"] = float(np.mean([x["hard"] for x in fold_res]))
    row["search_objective"] = float(np.mean([x["objective"] for x in fold_res]))
    rows.append(row)

    if i % 10 == 0:
        print(f"evaluated {i}/{sample_n}")

search_df = pd.DataFrame(rows).sort_values("search_objective", ascending=False).reset_index(drop=True)
print("top from search:")
display(search_df.head(15))

evaluated 10/180
evaluated 20/180
evaluated 30/180
evaluated 40/180
evaluated 50/180
evaluated 60/180
evaluated 70/180
evaluated 80/180
evaluated 90/180
evaluated 100/180
evaluated 110/180
evaluated 120/180
evaluated 130/180
evaluated 140/180
evaluated 150/180
evaluated 160/180
evaluated 170/180
evaluated 180/180
top from search:


,neg_per_pos,smooth_alpha,entity_w,assoc_w,overlap_w,jaccard_w,bi_key,ce_key,w_assoc,w_tfidf,w_bi,w_ce,search_overall,search_hard,search_objective
0,3,4.0,2.0,0.9,0.25,0.6,e5_body_query,ce_subj_body,1.0,0.2,1.2,1.0,0.848333,0.720238,0.758667
1,2,3.0,1.6,1.1,0.25,0.8,e5_subj_query,ce_body_subj,1.4,0.2,2.0,1.8,0.869167,0.705492,0.754595
2,2,4.0,1.2,0.7,0.35,0.8,e5_body_query,ce_body_subj,1.0,0.4,2.0,1.4,0.844167,0.713636,0.752795
3,2,3.0,1.2,0.9,0.25,0.4,e5_body_query,ce_subj_body,1.2,0.2,0.8,0.6,0.863333,0.705357,0.752750
4,2,4.0,2.0,0.7,0.45,0.6,mini,none,1.4,0.8,0.8,0.0,0.860000,0.703912,0.750739
5,2,4.0,1.6,1.1,0.25,0.6,e5_subj_query,ce_body_subj,1.2,0.4,1.6,0.6,0.869167,0.698235,0.749515
6,2,4.0,1.6,0.9,0.35,0.6,e5_subj_query,none,0.8,0.6,2.4,0.0,0.840000,0.709058,0.748341
7,4,4.0,1.2,1.1,0.45,0.6,e5_body_query,ce_subj_body,1.4,0.2,0.8,1.4,0.835833,0.709821,0.747625
8,2,3.0,1.6,0.9,0.45,0.4,e5_body_query,ce_body_subj,0.8,0.2,2.4,1.8,0.823333,0.714154,0.746908
9,3,4.0,1.6,0.7,0.45,0.8,e5_body_query,ce_subj_body,1.4,0.2,1.2,1.0,0.854167,0.700521,0.746615


In [6]:
top_k = 20
top_candidates = search_df.head(top_k).to_dict(orient="records")

holdout_rows = []
for cand in top_candidates:
    fold_res = [evaluate_params_on_fold(fold, cand) for fold in holdout_folds]

    row = dict(cand)
    row["holdout_overall"] = float(np.mean([x["overall"] for x in fold_res]))
    row["holdout_hard"] = float(np.mean([x["hard"] for x in fold_res]))
    row["holdout_objective"] = float(np.mean([x["objective"] for x in fold_res]))
    holdout_rows.append(row)

holdout_df = pd.DataFrame(holdout_rows).sort_values("holdout_objective", ascending=False).reset_index(drop=True)
display(holdout_df)

best = holdout_df.iloc[0].to_dict()
best_params = {
    "neg_per_pos": int(best["neg_per_pos"]),
    "smooth_alpha": float(best["smooth_alpha"]),
    "entity_w": float(best["entity_w"]),
    "assoc_w": float(best["assoc_w"]),
    "overlap_w": float(best["overlap_w"]),
    "jaccard_w": float(best["jaccard_w"]),
    "bi_key": str(best["bi_key"]),
    "ce_key": str(best["ce_key"]),
    "w_assoc": float(best["w_assoc"]),
    "w_tfidf": float(best["w_tfidf"]),
    "w_bi": float(best["w_bi"]),
    "w_ce": float(best["w_ce"]),
}

print("Best params selected:")
print(best_params)
print("Search objective:", round(float(best.get("search_objective", np.nan)), 4))
print("Holdout objective:", round(float(best.get("holdout_objective", np.nan)), 4))

,neg_per_pos,smooth_alpha,entity_w,assoc_w,overlap_w,jaccard_w,bi_key,ce_key,w_assoc,w_tfidf,w_bi,w_ce,search_overall,search_hard,search_objective,holdout_overall,holdout_hard,holdout_objective
0,3,5.0,1.2,1.1,0.45,0.8,mini,ce_subj_body,1.2,0.2,1.2,0.0,0.856667,0.692130,0.741491,0.838333,0.726073,0.759751
1,2,3.0,1.2,0.9,0.25,0.4,e5_body_query,ce_subj_body,1.2,0.2,0.8,0.6,0.863333,0.705357,0.752750,0.835556,0.681490,0.727710
2,3,3.0,1.2,0.9,0.25,0.8,mini,ce_body_subj,1.4,0.6,1.6,1.0,0.837500,0.692158,0.735761,0.826667,0.677859,0.722501
3,2,3.0,2.0,0.7,0.35,0.4,mini,ce_subj_body,1.0,0.8,2.0,0.6,0.818333,0.706250,0.739875,0.811111,0.675000,0.715833
4,3,4.0,2.0,0.9,0.25,0.6,e5_body_query,ce_subj_body,1.0,0.2,1.2,1.0,0.848333,0.720238,0.758667,0.826667,0.665530,0.713871
5,2,4.0,2.0,1.1,0.45,0.4,e5_body_query,ce_subj_body,1.2,0.8,1.6,0.0,0.846667,0.687840,0.735488,0.825000,0.663725,0.712108
6,2,3.0,1.6,1.1,0.25,0.6,mini,none,1.2,0.8,1.6,0.0,0.842500,0.702652,0.744606,0.828333,0.645423,0.700296
7,2,3.0,1.6,1.1,0.25,0.8,e5_subj_query,ce_body_subj,1.4,0.2,2.0,1.8,0.869167,0.705492,0.754595,0.825000,0.644444,0.698611
8,2,3.0,1.6,1.1,0.35,0.4,mini,ce_subj_body,0.8,0.4,0.8,1.4,0.825000,0.703125,0.739687,0.814444,0.646414,0.696823
9,3,5.0,1.2,0.9,0.25,0.6,e5_body_query,ce_subj_body,0.8,0.6,1.2,1.4,0.825833,0.700362,0.738004,0.803889,0.649229,0.695627


Best params selected:
{'neg_per_pos': 3, 'smooth_alpha': 5.0, 'entity_w': 1.2, 'assoc_w': 1.1, 'overlap_w': 0.45, 'jaccard_w': 0.8, 'bi_key': 'mini', 'ce_key': 'ce_subj_body', 'w_assoc': 1.2, 'w_tfidf': 0.2, 'w_bi': 1.2, 'w_ce': 0.0}
Search objective: 0.7415
Holdout objective: 0.7598


In [7]:
assoc_scores = build_association_scores(
    train_df,
    neg_per_pos=best_params["neg_per_pos"],
    smooth_alpha=best_params["smooth_alpha"],
    min_token_freq=8,
    min_log_ratio=0.02,
)

word_vec, char_vec = fit_tfidf_vectorizers(train_df)

selected_spec = BI_MODEL_SPECS[best_params["bi_key"]]
selected_bi_model = BI_MODELS[best_params["bi_key"]]

all_test_texts = sorted(
    set(test_df["body"].astype(str).tolist())
    | set(test_subjects_df["subject"].astype(str).tolist())
)

body_inputs = [apply_mode(t, selected_spec["body_mode"]) for t in all_test_texts]
subject_inputs = [apply_mode(t, selected_spec["subject_mode"]) for t in all_test_texts]

body_emb = selected_bi_model.encode(
    body_inputs,
    batch_size=128,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
subject_emb = selected_bi_model.encode(
    subject_inputs,
    batch_size=128,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

body_lookup = {t: body_emb[i] for i, t in enumerate(all_test_texts)}
subject_lookup = {t: subject_emb[i] for i, t in enumerate(all_test_texts)}

ce_lookup = {}
if best_params["ce_key"] != "none":
    pairs = []
    meta = []

    for block_id, body_group in test_df.groupby("block_id"):
        bodies = body_group.sort_values("body_index").reset_index(drop=True)
        subjects = (
            test_subjects_df[test_subjects_df["block_id"] == block_id]
            .sort_values("subject_letter")
            .reset_index(drop=True)
        )

        body_texts = bodies["body"].astype(str).tolist()
        subject_texts = subjects["subject"].astype(str).tolist()

        for i in range(4):
            for j in range(4):
                if best_params["ce_key"] == "ce_subj_body":
                    pairs.append((subject_texts[j], body_texts[i]))
                else:
                    pairs.append((body_texts[i], subject_texts[j]))
                meta.append((int(block_id), i, j))

    ce_scores = predict_cross_scores(pairs, batch_size=256, chunk_size=8192)

    for idx, (block_id, i, j) in enumerate(meta):
        if block_id not in ce_lookup:
            ce_lookup[block_id] = np.zeros((4, 4), dtype=np.float32)
        ce_lookup[block_id][i, j] = float(ce_scores[idx])

    for block_id in list(ce_lookup.keys()):
        ce_lookup[block_id] = zscore_matrix(ce_lookup[block_id])


out_rows = []
for block_id, body_group in test_df.groupby("block_id"):
    bodies = body_group.sort_values("body_index").reset_index(drop=True)
    subjects = (
        test_subjects_df[test_subjects_df["block_id"] == block_id]
        .sort_values("subject_letter")
        .reset_index(drop=True)
    )

    body_texts = bodies["body"].astype(str).tolist()
    subject_texts = subjects["subject"].astype(str).tolist()
    subject_letters = subjects["subject_letter"].tolist()

    body_tokens = [token_set(x) for x in body_texts]
    subj_tokens = [token_set(x) for x in subject_texts]
    body_entities = [entity_set(x) for x in body_texts]
    subj_entities = [entity_set(x) for x in subject_texts]

    assoc_raw = association_matrix(
        body_tokens,
        subj_tokens,
        body_entities,
        subj_entities,
        assoc_scores=assoc_scores,
        entity_w=best_params["entity_w"],
        assoc_w=best_params["assoc_w"],
        overlap_w=best_params["overlap_w"],
        jaccard_w=best_params["jaccard_w"],
    )
    assoc_z = zscore_matrix(assoc_raw)

    tfidf_z = zscore_matrix(tfidf_matrix(body_texts, subject_texts, word_vec, char_vec))

    b = np.vstack([body_lookup[t] for t in body_texts]).astype(np.float32)
    s = np.vstack([subject_lookup[t] for t in subject_texts]).astype(np.float32)
    bi_z = zscore_matrix(cosine_matrix(b, s))

    ce_z = ce_lookup.get(int(block_id), np.zeros((4, 4), dtype=np.float32))

    final_mat = (
        best_params["w_assoc"] * assoc_z
        + best_params["w_tfidf"] * tfidf_z
        + best_params["w_bi"] * bi_z
        + best_params["w_ce"] * ce_z
    )

    assignment, _ = best_assignment(final_mat)

    for i, subj_col in enumerate(assignment):
        out_rows.append(
            {
                "block_id": int(block_id),
                "body_index": int(bodies.loc[i, "body_index"]),
                "assigned_subject": subject_letters[subj_col],
            }
        )

submission = pd.DataFrame(out_rows).sort_values(["block_id", "body_index"]).reset_index(drop=True)
out_path = WORK_DIR / "submission.csv"
submission.to_csv(out_path, index=False)

best_payload = dict(best_params)
best_payload["bi_model_name"] = selected_spec["name"]
best_payload["cross_model_name"] = CROSS_MODEL_NAME if best_params["ce_key"] != "none" else "none"

params_path = WORK_DIR / "best_params.json"
with open(params_path, "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)

print("Saved submission to:", out_path)
print("Saved params to:", params_path)
print("Selected bi model:", selected_spec["name"])
print("Selected cross mode:", best_params["ce_key"])
display(submission.head(12))

Saved submission to: /kaggle/working/submission.csv
Saved params to: /kaggle/working/best_params.json
Selected bi model: sentence-transformers/all-MiniLM-L6-v2
Selected cross mode: ce_subj_body


,block_id,body_index,assigned_subject
0,1,0,D
1,1,1,B
2,1,2,A
3,1,3,C
4,2,0,A
5,2,1,C
6,2,2,B
7,2,3,D
8,3,0,A
9,3,1,C


In [8]:
sub = pd.read_csv(WORK_DIR / "submission.csv")

print("submission shape:", sub.shape)
print("expected keys:", test_df.shape[0])
print("columns:", list(sub.columns))
print("missing values:", sub.isna().sum().to_dict())

keys_ok = set(map(tuple, sub[["block_id", "body_index"]].to_numpy())) == set(
    map(tuple, test_df[["block_id", "body_index"]].to_numpy())
)
print(
    "all keys present exactly once:",
    keys_ok and sub[["block_id", "body_index"]].drop_duplicates().shape[0] == test_df.shape[0],
)
print("assigned_subject unique:", sorted(sub["assigned_subject"].unique().tolist()))

repeat_blocks = int((sub.groupby("block_id")["assigned_subject"].nunique() < 4).sum())
print("blocks with repeated letters:", repeat_blocks)

submission shape: (820, 3)
expected keys: 820
columns: ['block_id', 'body_index', 'assigned_subject']
missing values: {'block_id': 0, 'body_index': 0, 'assigned_subject': 0}
all keys present exactly once: True
assigned_subject unique: ['A', 'B', 'C', 'D']
blocks with repeated letters: 0


In [9]:
# Round-2 calibration diagnostic: compare stable candidate families on all folds.
all_folds = search_folds + holdout_folds


def eval_candidate(candidate: dict) -> dict:
    fold_res = [evaluate_params_on_fold(f, candidate) for f in all_folds]

    overall = np.array([x["overall"] for x in fold_res], dtype=np.float64)
    hard = np.array([x["hard"] for x in fold_res], dtype=np.float64)

    mean_overall = float(overall.mean())
    mean_hard = float(hard.mean())
    worst_overall = float(overall.min())

    robust = 0.5 * mean_overall + 0.3 * mean_hard + 0.2 * worst_overall

    out = dict(candidate)
    out["mean_overall"] = mean_overall
    out["mean_hard"] = mean_hard
    out["worst_overall"] = worst_overall
    out["robust"] = robust
    return out


candidates = [
    {
        "name": "assoc_tfidf_only",
        "neg_per_pos": 2,
        "smooth_alpha": 4.0,
        "entity_w": 1.6,
        "assoc_w": 0.8,
        "overlap_w": 0.35,
        "jaccard_w": 0.6,
        "bi_key": "mini",
        "ce_key": "none",
        "w_assoc": 1.0,
        "w_tfidf": 1.0,
        "w_bi": 0.0,
        "w_ce": 0.0,
    },
    {
        "name": "current_shipd",
        "neg_per_pos": 3,
        "smooth_alpha": 5.0,
        "entity_w": 1.2,
        "assoc_w": 1.1,
        "overlap_w": 0.45,
        "jaccard_w": 0.8,
        "bi_key": "mini",
        "ce_key": "ce_subj_body",
        "w_assoc": 1.2,
        "w_tfidf": 0.2,
        "w_bi": 1.2,
        "w_ce": 0.0,
    },
    {
        "name": "mini_plus_ce_low",
        "neg_per_pos": 3,
        "smooth_alpha": 5.0,
        "entity_w": 1.2,
        "assoc_w": 1.1,
        "overlap_w": 0.45,
        "jaccard_w": 0.8,
        "bi_key": "mini",
        "ce_key": "ce_subj_body",
        "w_assoc": 1.1,
        "w_tfidf": 0.2,
        "w_bi": 1.0,
        "w_ce": 0.6,
    },
    {
        "name": "mini_plus_ce_mid",
        "neg_per_pos": 3,
        "smooth_alpha": 5.0,
        "entity_w": 1.2,
        "assoc_w": 1.1,
        "overlap_w": 0.45,
        "jaccard_w": 0.8,
        "bi_key": "mini",
        "ce_key": "ce_subj_body",
        "w_assoc": 1.0,
        "w_tfidf": 0.2,
        "w_bi": 1.0,
        "w_ce": 1.0,
    },
    {
        "name": "mini_plus_ce_high",
        "neg_per_pos": 3,
        "smooth_alpha": 5.0,
        "entity_w": 1.2,
        "assoc_w": 1.1,
        "overlap_w": 0.45,
        "jaccard_w": 0.8,
        "bi_key": "mini",
        "ce_key": "ce_subj_body",
        "w_assoc": 0.9,
        "w_tfidf": 0.2,
        "w_bi": 0.9,
        "w_ce": 1.4,
    },
    {
        "name": "e5_query_passage",
        "neg_per_pos": 3,
        "smooth_alpha": 4.0,
        "entity_w": 1.6,
        "assoc_w": 1.0,
        "overlap_w": 0.35,
        "jaccard_w": 0.6,
        "bi_key": "e5_subj_query",
        "ce_key": "none",
        "w_assoc": 1.0,
        "w_tfidf": 0.4,
        "w_bi": 1.6,
        "w_ce": 0.0,
    },
    {
        "name": "ce_only_subj_body",
        "neg_per_pos": 3,
        "smooth_alpha": 4.0,
        "entity_w": 1.2,
        "assoc_w": 0.8,
        "overlap_w": 0.25,
        "jaccard_w": 0.4,
        "bi_key": "mini",
        "ce_key": "ce_subj_body",
        "w_assoc": 0.0,
        "w_tfidf": 0.0,
        "w_bi": 0.0,
        "w_ce": 1.8,
    },
]

rows = [eval_candidate(c) for c in candidates]
calib_df = pd.DataFrame(rows).sort_values("robust", ascending=False).reset_index(drop=True)

show_cols = [
    "name",
    "mean_overall",
    "mean_hard",
    "worst_overall",
    "robust",
    "bi_key",
    "ce_key",
    "w_assoc",
    "w_tfidf",
    "w_bi",
    "w_ce",
]

print("Round-2 calibration ranking:")
display(calib_df[show_cols])

Round-2 calibration ranking:


,name,mean_overall,mean_hard,worst_overall,robust,bi_key,ce_key,w_assoc,w_tfidf,w_bi,w_ce
0,current_shipd,0.845667,0.712496,0.821667,0.800915,mini,ce_subj_body,1.2,0.2,1.2,0.0
1,mini_plus_ce_low,0.834000,0.648374,0.821667,0.775846,mini,ce_subj_body,1.1,0.2,1.0,0.6
2,mini_plus_ce_mid,0.826000,0.648176,0.805000,0.768453,mini,ce_subj_body,1.0,0.2,1.0,1.0
3,mini_plus_ce_high,0.816667,0.623757,0.800000,0.755460,mini,ce_subj_body,0.9,0.2,0.9,1.4
4,e5_query_passage,0.833667,0.556508,0.796667,0.743119,e5_subj_query,none,1.0,0.4,1.6,0.0
5,assoc_tfidf_only,0.805333,0.612399,0.753333,0.737053,mini,none,1.0,1.0,0.0,0.0
6,ce_only_subj_body,0.664333,0.627090,0.648333,0.649960,mini,ce_subj_body,0.0,0.0,0.0,1.8


In [10]:
# Round-2 tougher validation: template-holdout folds to reduce train/val leakage.
SKELETON_KEEP = {
    "budget", "contract", "approval", "document", "requirements", "compliance",
    "meeting", "workshop", "call", "sync", "review", "agenda",
    "progress", "migration", "rollout", "delivery", "milestone", "release",
    "proposal", "pricing", "partnership", "solution", "demo", "renewal",
    "issue", "ticket", "resolution", "incident", "escalation", "support",
    "status", "timeline", "risk", "action", "plan", "update",
}


def skeleton(text: str) -> str:
    s = str(text)
    s = ENTITY_RE.sub(" ENT ", s)
    s = re.sub(r"\$?\b\d[\d,]*(?:\.\d+)?\b", " NUM ", s)
    toks = [t.lower() for t in TOKEN_RE.findall(s)]

    out = []
    for t in toks:
        if t in STOPWORDS:
            continue
        if t in {"ent", "num"}:
            out.append(t)
        elif t in SKELETON_KEEP:
            out.append(t)
        else:
            out.append("x")

    # Compress repeated placeholders to keep keys short but expressive.
    compact = []
    for t in out:
        if compact and compact[-1] == "x" and t == "x":
            continue
        compact.append(t)

    return " ".join(compact[:16])


def split_train_val_template(df: pd.DataFrame, seed: int, train_frac: float = 0.8) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)

    tr_parts = []
    va_parts = []

    for cat, g in df.groupby("category", sort=False):
        tmp = g.copy()
        tmp["sk_body"] = tmp["body"].map(skeleton)
        tmp["sk_subj"] = tmp["subject"].map(skeleton)
        tmp["tpl"] = tmp["sk_body"] + " || " + tmp["sk_subj"]

        groups = tmp["tpl"].drop_duplicates().tolist()
        rng.shuffle(groups)

        cut = max(1, int(len(groups) * train_frac))
        train_groups = set(groups[:cut])

        tr_cat = tmp[tmp["tpl"].isin(train_groups)].drop(columns=["sk_body", "sk_subj", "tpl"])
        va_cat = tmp[~tmp["tpl"].isin(train_groups)].drop(columns=["sk_body", "sk_subj", "tpl"])

        # Fallback if template split is too harsh in small categories.
        if len(va_cat) < 8:
            idx = g.index.to_numpy().copy()
            rng.shuffle(idx)
            cut_idx = int(len(idx) * train_frac)
            tr_cat = g.loc[idx[:cut_idx]]
            va_cat = g.loc[idx[cut_idx:]]

        tr_parts.append(tr_cat)
        va_parts.append(va_cat)

    tr = pd.concat(tr_parts, ignore_index=True)
    va = pd.concat(va_parts, ignore_index=True)
    return tr, va


def build_fold_cache_from_split(seed: int, tr: pd.DataFrame, va: pd.DataFrame) -> dict:
    raw_blocks = make_val_blocks(va, seed=seed + 1999)

    word_vec, char_vec = fit_tfidf_vectorizers(tr)

    unique_texts = sorted({txt for _, bodies, subjects, _ in raw_blocks for txt in (bodies + subjects)})

    bi_lookup = {}
    for key, spec in BI_MODEL_SPECS.items():
        model = BI_MODELS[key]

        body_inputs = [apply_mode(t, spec["body_mode"]) for t in unique_texts]
        subj_inputs = [apply_mode(t, spec["subject_mode"]) for t in unique_texts]

        body_emb = model.encode(
            body_inputs,
            batch_size=128,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        subj_emb = model.encode(
            subj_inputs,
            batch_size=128,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        bi_lookup[key] = {
            "body": {t: body_emb[i] for i, t in enumerate(unique_texts)},
            "subject": {t: subj_emb[i] for i, t in enumerate(unique_texts)},
        }

    sb_pairs = []
    bs_pairs = []
    meta = []
    for blk_idx, (_, bodies, subjects, _) in enumerate(raw_blocks):
        for i in range(4):
            for j in range(4):
                sb_pairs.append((subjects[j], bodies[i]))
                bs_pairs.append((bodies[i], subjects[j]))
                meta.append((blk_idx, i, j))

    sb_scores = predict_cross_scores(sb_pairs, batch_size=256, chunk_size=4096)
    bs_scores = predict_cross_scores(bs_pairs, batch_size=256, chunk_size=4096)

    ce_sb = [np.zeros((4, 4), dtype=np.float32) for _ in raw_blocks]
    ce_bs = [np.zeros((4, 4), dtype=np.float32) for _ in raw_blocks]

    for idx, (blk_idx, i, j) in enumerate(meta):
        ce_sb[blk_idx][i, j] = float(sb_scores[idx])
        ce_bs[blk_idx][i, j] = float(bs_scores[idx])

    blocks = []
    for blk_idx, (cat, bodies, subjects, gt) in enumerate(raw_blocks):
        tfidf_z = zscore_matrix(tfidf_matrix(bodies, subjects, word_vec, char_vec))

        bi_z = {}
        for key in BI_MODEL_SPECS:
            b = np.vstack([bi_lookup[key]["body"][t] for t in bodies]).astype(np.float32)
            s = np.vstack([bi_lookup[key]["subject"][t] for t in subjects]).astype(np.float32)
            bi_z[key] = zscore_matrix(cosine_matrix(b, s))

        ce_z = {
            "none": np.zeros((4, 4), dtype=np.float32),
            "ce_subj_body": zscore_matrix(ce_sb[blk_idx]),
            "ce_body_subj": zscore_matrix(ce_bs[blk_idx]),
        }

        blocks.append(
            BlockCache(
                category=cat,
                body_texts=bodies,
                subject_texts=subjects,
                body_tokens=[token_set(x) for x in bodies],
                subj_tokens=[token_set(x) for x in subjects],
                body_entities=[entity_set(x) for x in bodies],
                subj_entities=[entity_set(x) for x in subjects],
                tfidf_z=tfidf_z,
                bi_z=bi_z,
                ce_z=ce_z,
                gt=gt,
            )
        )

    return {
        "seed": seed,
        "train_df": tr,
        "blocks": blocks,
    }


template_seeds = [111, 222, 333]
template_folds = []
for s in template_seeds:
    tr_t, va_t = split_train_val_template(train_df, seed=s, train_frac=0.8)
    fold = build_fold_cache_from_split(seed=s, tr=tr_t, va=va_t)
    template_folds.append(fold)
    print(f"template-fold seed={s} train={tr_t.shape} val={va_t.shape} blocks={len(fold['blocks'])}")

template-fold seed=111 train=(2458, 4) val=(542, 4) blocks=134
template-fold seed=222 train=(2416, 4) val=(584, 4) blocks=144
template-fold seed=333 train=(2408, 4) val=(592, 4) blocks=146


In [11]:
# Round-2 constrained search on tougher validation.


def summarize_candidate_on_folds(candidate: dict, folds: list) -> dict:
    fold_res = [evaluate_params_on_fold(f, candidate) for f in folds]

    overall = np.array([x["overall"] for x in fold_res], dtype=np.float64)
    hard = np.array([x["hard"] for x in fold_res], dtype=np.float64)

    mean_overall = float(overall.mean())
    mean_hard = float(hard.mean())
    worst_overall = float(overall.min())
    robust = 0.5 * mean_overall + 0.3 * mean_hard + 0.2 * worst_overall

    return {
        "mean_overall": mean_overall,
        "mean_hard": mean_hard,
        "worst_overall": worst_overall,
        "robust": robust,
    }


rng = np.random.default_rng(9090)

pool2 = {
    "neg_per_pos": [2, 3, 4, 5],
    "smooth_alpha": [2.0, 3.0, 4.0, 5.0, 6.0],
    "entity_w": [1.0, 1.2, 1.6, 2.0],
    "assoc_w": [0.7, 0.9, 1.1, 1.3],
    "overlap_w": [0.2, 0.35, 0.45],
    "jaccard_w": [0.4, 0.6, 0.8],
    "bi_key": ["mini", "e5_subj_query", "e5_body_query"],
    "ce_key": ["none", "ce_subj_body", "ce_body_subj"],
    "w_assoc": [0.6, 0.9, 1.2, 1.5],
    "w_tfidf": [0.0, 0.2, 0.4, 0.8, 1.2],
    "w_bi": [0.0, 0.6, 1.0, 1.4, 1.8, 2.2],
    "w_ce": [0.0, 0.4, 0.8, 1.2, 1.6, 2.0],
}

keys2 = [
    "neg_per_pos",
    "smooth_alpha",
    "entity_w",
    "assoc_w",
    "overlap_w",
    "jaccard_w",
    "bi_key",
    "ce_key",
    "w_assoc",
    "w_tfidf",
    "w_bi",
    "w_ce",
]


def sample_candidate() -> dict:
    c = {
        "neg_per_pos": int(rng.choice(pool2["neg_per_pos"])),
        "smooth_alpha": float(rng.choice(pool2["smooth_alpha"])),
        "entity_w": float(rng.choice(pool2["entity_w"])),
        "assoc_w": float(rng.choice(pool2["assoc_w"])),
        "overlap_w": float(rng.choice(pool2["overlap_w"])),
        "jaccard_w": float(rng.choice(pool2["jaccard_w"])),
        "bi_key": str(rng.choice(pool2["bi_key"])),
        "ce_key": str(rng.choice(pool2["ce_key"])),
        "w_assoc": float(rng.choice(pool2["w_assoc"])),
        "w_tfidf": float(rng.choice(pool2["w_tfidf"])),
        "w_bi": float(rng.choice(pool2["w_bi"])),
        "w_ce": float(rng.choice(pool2["w_ce"])),
    }

    if c["ce_key"] == "none":
        c["w_ce"] = 0.0
    else:
        if c["w_ce"] <= 0.0:
            c["w_ce"] = 0.8

    if c["w_tfidf"] == 0.0 and c["w_bi"] == 0.0 and c["w_ce"] == 0.0:
        c["w_bi"] = 1.0

    return c


seeded_candidates = [
    {
        "neg_per_pos": 3,
        "smooth_alpha": 5.0,
        "entity_w": 1.2,
        "assoc_w": 1.1,
        "overlap_w": 0.45,
        "jaccard_w": 0.8,
        "bi_key": "mini",
        "ce_key": "ce_subj_body",
        "w_assoc": 1.2,
        "w_tfidf": 0.2,
        "w_bi": 1.2,
        "w_ce": 0.0,
    },
    {
        "neg_per_pos": 3,
        "smooth_alpha": 5.0,
        "entity_w": 1.2,
        "assoc_w": 1.1,
        "overlap_w": 0.45,
        "jaccard_w": 0.8,
        "bi_key": "mini",
        "ce_key": "ce_subj_body",
        "w_assoc": 1.0,
        "w_tfidf": 0.2,
        "w_bi": 1.0,
        "w_ce": 1.0,
    },
    {
        "neg_per_pos": 2,
        "smooth_alpha": 4.0,
        "entity_w": 1.6,
        "assoc_w": 0.8,
        "overlap_w": 0.35,
        "jaccard_w": 0.6,
        "bi_key": "mini",
        "ce_key": "none",
        "w_assoc": 1.0,
        "w_tfidf": 1.0,
        "w_bi": 0.0,
        "w_ce": 0.0,
    },
]

candidates2 = []
seen = set()
for c in seeded_candidates:
    k = tuple(c[x] for x in keys2)
    if k not in seen:
        seen.add(k)
        candidates2.append(c)

sample_n2 = 260
while len(candidates2) < sample_n2:
    c = sample_candidate()
    k = tuple(c[x] for x in keys2)
    if k in seen:
        continue
    seen.add(k)
    candidates2.append(c)

rows2 = []
for i, cand in enumerate(candidates2, 1):
    tpl = summarize_candidate_on_folds(cand, template_folds)
    std = summarize_candidate_on_folds(cand, holdout_folds)

    row = dict(cand)
    row["tpl_robust"] = tpl["robust"]
    row["tpl_mean"] = tpl["mean_overall"]
    row["tpl_hard"] = tpl["mean_hard"]
    row["tpl_worst"] = tpl["worst_overall"]

    row["std_robust"] = std["robust"]
    row["std_mean"] = std["mean_overall"]
    row["std_hard"] = std["mean_hard"]
    row["std_worst"] = std["worst_overall"]

    row["combined_robust"] = 0.7 * row["tpl_robust"] + 0.3 * row["std_robust"]
    rows2.append(row)

    if i % 20 == 0:
        print(f"evaluated {i}/{len(candidates2)}")

round2_df = pd.DataFrame(rows2).sort_values("combined_robust", ascending=False).reset_index(drop=True)

print("Round-2 top candidates (template-aware robust ranking):")
display(
    round2_df[
        [
            "combined_robust",
            "tpl_robust",
            "std_robust",
            "tpl_mean",
            "tpl_hard",
            "tpl_worst",
            "neg_per_pos",
            "smooth_alpha",
            "entity_w",
            "assoc_w",
            "overlap_w",
            "jaccard_w",
            "bi_key",
            "ce_key",
            "w_assoc",
            "w_tfidf",
            "w_bi",
            "w_ce",
        ]
    ].head(20)
)

evaluated 20/260
evaluated 40/260
evaluated 60/260
evaluated 80/260
evaluated 100/260
evaluated 120/260
evaluated 140/260
evaluated 160/260
evaluated 180/260
evaluated 200/260
evaluated 220/260
evaluated 240/260
evaluated 260/260
Round-2 top candidates (template-aware robust ranking):


,combined_robust,tpl_robust,std_robust,tpl_mean,tpl_hard,tpl_worst,neg_per_pos,smooth_alpha,entity_w,assoc_w,overlap_w,jaccard_w,bi_key,ce_key,w_assoc,w_tfidf,w_bi,w_ce
0,0.753446,0.743999,0.775489,0.777138,0.678550,0.759328,4,4.0,1.6,1.3,0.45,0.6,mini,ce_body_subj,0.9,0.0,1.0,0.4
1,0.749342,0.735410,0.781850,0.770482,0.670961,0.744403,2,6.0,1.2,1.1,0.35,0.4,mini,ce_subj_body,1.2,0.2,1.0,2.0
2,0.741402,0.730785,0.766176,0.767714,0.657671,0.748134,4,3.0,1.6,1.1,0.35,0.6,mini,ce_body_subj,0.9,0.2,0.6,1.2
3,0.740935,0.722869,0.783088,0.757882,0.651401,0.742537,2,6.0,1.6,0.7,0.20,0.8,e5_subj_query,none,0.6,0.0,0.6,0.0
4,0.740492,0.714422,0.801322,0.762018,0.611377,0.750000,3,5.0,1.2,1.1,0.45,0.8,mini,ce_subj_body,1.2,0.2,1.2,0.0
5,0.739323,0.729827,0.761481,0.768907,0.651245,0.750000,2,6.0,1.6,1.1,0.20,0.8,e5_subj_query,none,0.6,0.2,1.8,0.0
6,0.739113,0.721732,0.779668,0.756507,0.653636,0.736940,4,2.0,1.6,0.9,0.20,0.6,mini,ce_body_subj,1.2,0.4,0.6,1.2
7,0.739046,0.725550,0.770537,0.764557,0.651701,0.738806,2,6.0,1.2,0.7,0.35,0.8,mini,none,1.2,0.2,1.8,0.0
8,0.736904,0.716254,0.785086,0.758402,0.643411,0.720149,2,6.0,1.6,0.7,0.35,0.8,e5_body_query,ce_subj_body,0.6,0.0,0.6,0.8
9,0.736696,0.711875,0.794612,0.763046,0.618585,0.723881,2,2.0,2.0,1.3,0.20,0.6,mini,ce_subj_body,1.2,0.2,1.4,0.8


In [12]:
top_cols = [
    "combined_robust",
    "tpl_robust",
    "std_robust",
    "tpl_mean",
    "tpl_hard",
    "tpl_worst",
    "neg_per_pos",
    "smooth_alpha",
    "entity_w",
    "assoc_w",
    "overlap_w",
    "jaccard_w",
    "bi_key",
    "ce_key",
    "w_assoc",
    "w_tfidf",
    "w_bi",
    "w_ce",
]
print(round2_df[top_cols].head(10).to_string(index=False))

 combined_robust  tpl_robust  std_robust  tpl_mean  tpl_hard  tpl_worst  neg_per_pos  smooth_alpha  entity_w  assoc_w  overlap_w  jaccard_w        bi_key       ce_key  w_assoc  w_tfidf  w_bi  w_ce
        0.753446    0.743999    0.775489  0.777138  0.678550   0.759328            4           4.0       1.6      1.3       0.45        0.6          mini ce_body_subj      0.9      0.0   1.0   0.4
        0.749342    0.735410    0.781850  0.770482  0.670961   0.744403            2           6.0       1.2      1.1       0.35        0.4          mini ce_subj_body      1.2      0.2   1.0   2.0
        0.741402    0.730785    0.766176  0.767714  0.657671   0.748134            4           3.0       1.6      1.1       0.35        0.6          mini ce_body_subj      0.9      0.2   0.6   1.2
        0.740935    0.722869    0.783088  0.757882  0.651401   0.742537            2           6.0       1.6      0.7       0.20        0.8 e5_subj_query         none      0.6      0.0   0.6   0.0
        0.74049

In [13]:
param_keys = [
    "neg_per_pos",
    "smooth_alpha",
    "entity_w",
    "assoc_w",
    "overlap_w",
    "jaccard_w",
    "bi_key",
    "ce_key",
    "w_assoc",
    "w_tfidf",
    "w_bi",
    "w_ce",
]


def row_to_params(row: pd.Series) -> dict:
    return {
        "neg_per_pos": int(row["neg_per_pos"]),
        "smooth_alpha": float(row["smooth_alpha"]),
        "entity_w": float(row["entity_w"]),
        "assoc_w": float(row["assoc_w"]),
        "overlap_w": float(row["overlap_w"]),
        "jaccard_w": float(row["jaccard_w"]),
        "bi_key": str(row["bi_key"]),
        "ce_key": str(row["ce_key"]),
        "w_assoc": float(row["w_assoc"]),
        "w_tfidf": float(row["w_tfidf"]),
        "w_bi": float(row["w_bi"]),
        "w_ce": float(row["w_ce"]),
    }


word_vec_full, char_vec_full = fit_tfidf_vectorizers(train_df)
all_test_texts = sorted(set(test_df["body"].astype(str).tolist()) | set(test_subjects_df["subject"].astype(str).tolist()))

BI_INFER_CACHE = {}
CE_INFER_CACHE = {}
ASSOC_INFER_CACHE = {}


def get_bi_lookup_for_key(bi_key: str):
    if bi_key in BI_INFER_CACHE:
        return BI_INFER_CACHE[bi_key]

    spec = BI_MODEL_SPECS[bi_key]
    model = BI_MODELS[bi_key]

    body_inputs = [apply_mode(t, spec["body_mode"]) for t in all_test_texts]
    subject_inputs = [apply_mode(t, spec["subject_mode"]) for t in all_test_texts]

    body_emb = model.encode(
        body_inputs,
        batch_size=128,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    subject_emb = model.encode(
        subject_inputs,
        batch_size=128,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    lookup = {
        "body": {t: body_emb[i] for i, t in enumerate(all_test_texts)},
        "subject": {t: subject_emb[i] for i, t in enumerate(all_test_texts)},
        "model_name": spec["name"],
    }
    BI_INFER_CACHE[bi_key] = lookup
    return lookup


def get_ce_lookup(ce_key: str):
    if ce_key == "none":
        return {}
    if ce_key in CE_INFER_CACHE:
        return CE_INFER_CACHE[ce_key]

    pairs = []
    meta = []

    for block_id, body_group in test_df.groupby("block_id"):
        bodies = body_group.sort_values("body_index").reset_index(drop=True)
        subjects = (
            test_subjects_df[test_subjects_df["block_id"] == block_id]
            .sort_values("subject_letter")
            .reset_index(drop=True)
        )

        body_texts = bodies["body"].astype(str).tolist()
        subject_texts = subjects["subject"].astype(str).tolist()

        for i in range(4):
            for j in range(4):
                if ce_key == "ce_subj_body":
                    pairs.append((subject_texts[j], body_texts[i]))
                else:
                    pairs.append((body_texts[i], subject_texts[j]))
                meta.append((int(block_id), i, j))

    ce_scores = predict_cross_scores(pairs, batch_size=256, chunk_size=8192)

    out = {}
    for idx, (block_id, i, j) in enumerate(meta):
        if block_id not in out:
            out[block_id] = np.zeros((4, 4), dtype=np.float32)
        out[block_id][i, j] = float(ce_scores[idx])

    for block_id in list(out.keys()):
        out[block_id] = zscore_matrix(out[block_id])

    CE_INFER_CACHE[ce_key] = out
    return out


def get_assoc_scores_for_params(params: dict):
    k = (params["neg_per_pos"], params["smooth_alpha"])
    if k in ASSOC_INFER_CACHE:
        return ASSOC_INFER_CACHE[k]

    scores = build_association_scores(
        train_df,
        neg_per_pos=params["neg_per_pos"],
        smooth_alpha=params["smooth_alpha"],
        min_token_freq=8,
        min_log_ratio=0.02,
    )
    ASSOC_INFER_CACHE[k] = scores
    return scores


def generate_submission_for_params(params: dict) -> pd.DataFrame:
    assoc_scores = get_assoc_scores_for_params(params)
    bi_lookup = get_bi_lookup_for_key(params["bi_key"])
    ce_lookup = get_ce_lookup(params["ce_key"])

    out_rows = []

    for block_id, body_group in test_df.groupby("block_id"):
        bodies = body_group.sort_values("body_index").reset_index(drop=True)
        subjects = (
            test_subjects_df[test_subjects_df["block_id"] == block_id]
            .sort_values("subject_letter")
            .reset_index(drop=True)
        )

        body_texts = bodies["body"].astype(str).tolist()
        subject_texts = subjects["subject"].astype(str).tolist()
        subject_letters = subjects["subject_letter"].tolist()

        body_tokens = [token_set(x) for x in body_texts]
        subj_tokens = [token_set(x) for x in subject_texts]
        body_entities = [entity_set(x) for x in body_texts]
        subj_entities = [entity_set(x) for x in subject_texts]

        assoc_raw = association_matrix(
            body_tokens,
            subj_tokens,
            body_entities,
            subj_entities,
            assoc_scores=assoc_scores,
            entity_w=params["entity_w"],
            assoc_w=params["assoc_w"],
            overlap_w=params["overlap_w"],
            jaccard_w=params["jaccard_w"],
        )
        assoc_z = zscore_matrix(assoc_raw)

        tfidf_z = zscore_matrix(tfidf_matrix(body_texts, subject_texts, word_vec_full, char_vec_full))

        b = np.vstack([bi_lookup["body"][t] for t in body_texts]).astype(np.float32)
        s = np.vstack([bi_lookup["subject"][t] for t in subject_texts]).astype(np.float32)
        bi_z = zscore_matrix(cosine_matrix(b, s))

        ce_z = ce_lookup.get(int(block_id), np.zeros((4, 4), dtype=np.float32))

        final_mat = (
            params["w_assoc"] * assoc_z
            + params["w_tfidf"] * tfidf_z
            + params["w_bi"] * bi_z
            + params["w_ce"] * ce_z
        )

        assignment, _ = best_assignment(final_mat)

        for i, subj_col in enumerate(assignment):
            out_rows.append(
                {
                    "block_id": int(block_id),
                    "body_index": int(bodies.loc[i, "body_index"]),
                    "assigned_subject": subject_letters[subj_col],
                }
            )

    return pd.DataFrame(out_rows).sort_values(["block_id", "body_index"]).reset_index(drop=True)


top3_rows = round2_df.head(3)
export_rows = []

for rank_idx in range(3):
    row = top3_rows.iloc[rank_idx]
    params = row_to_params(row)

    sub_df = generate_submission_for_params(params)
    sub_path = WORK_DIR / f"submission_candidate_{rank_idx+1}.csv"
    cfg_path = WORK_DIR / f"best_params_candidate_{rank_idx+1}.json"

    sub_df.to_csv(sub_path, index=False)

    payload = dict(params)
    payload["bi_model_name"] = BI_MODEL_SPECS[params["bi_key"]]["name"]
    payload["cross_model_name"] = CROSS_MODEL_NAME if params["ce_key"] != "none" else "none"
    payload["combined_robust"] = float(row["combined_robust"])
    payload["tpl_robust"] = float(row["tpl_robust"])
    payload["std_robust"] = float(row["std_robust"])

    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    export_rows.append(
        {
            "rank": rank_idx + 1,
            "submission": str(sub_path),
            "params": str(cfg_path),
            "combined_robust": float(row["combined_robust"]),
            "tpl_robust": float(row["tpl_robust"]),
            "std_robust": float(row["std_robust"]),
            "bi_key": params["bi_key"],
            "ce_key": params["ce_key"],
            "w_ce": params["w_ce"],
        }
    )

# Promote candidate 1 as default notebook output.
default_sub = WORK_DIR / "submission.csv"
default_cfg = WORK_DIR / "best_params.json"
pd.read_csv(WORK_DIR / "submission_candidate_1.csv").to_csv(default_sub, index=False)
with open(WORK_DIR / "best_params_candidate_1.json", "r", encoding="utf-8") as f:
    best_payload = json.load(f)
with open(default_cfg, "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)

print("Export complete:")
for r in export_rows:
    print(r)

print("\nDefaulted to:", default_sub)
print("Default params:", default_cfg)

Export complete:
{'rank': 1, 'submission': '/kaggle/working/submission_candidate_1.csv', 'params': '/kaggle/working/best_params_candidate_1.json', 'combined_robust': 0.753446275190438, 'tpl_robust': 0.7439994668051793, 'std_robust': 0.7754888280893752, 'bi_key': 'mini', 'ce_key': 'ce_body_subj', 'w_ce': 0.4}
{'rank': 2, 'submission': '/kaggle/working/submission_candidate_2.csv', 'params': '/kaggle/working/best_params_candidate_2.json', 'combined_robust': 0.7493420721477448, 'tpl_robust': 0.7354099199180237, 'std_robust': 0.7818504273504274, 'bi_key': 'mini', 'ce_key': 'ce_subj_body', 'w_ce': 2.0}
{'rank': 3, 'submission': '/kaggle/working/submission_candidate_3.csv', 'params': '/kaggle/working/best_params_candidate_3.json', 'combined_robust': 0.7414024187681081, 'tpl_robust': 0.7307852466709552, 'std_robust': 0.7661758203281313, 'bi_key': 'mini', 'ce_key': 'ce_body_subj', 'w_ce': 1.2}

Defaulted to: /kaggle/working/submission.csv
Default params: /kaggle/working/best_params.json


In [14]:
from pathlib import Path

LOCAL_WORK_DIR = Path("/home/nothing/shipd_problems/Email Subject Line Matching/working")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "submission.csv",
    "best_params.json",
    "submission_candidate_1.csv",
    "submission_candidate_2.csv",
    "submission_candidate_3.csv",
    "best_params_candidate_1.json",
    "best_params_candidate_2.json",
    "best_params_candidate_3.json",
]

for name in files_to_copy:
    src = WORK_DIR / name
    dst = LOCAL_WORK_DIR / name

    if name.endswith(".csv"):
        pd.read_csv(src).to_csv(dst, index=False)
    else:
        with open(src, "r", encoding="utf-8") as f:
            obj = json.load(f)
        with open(dst, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2)

print("Copied files to", LOCAL_WORK_DIR)
for name in files_to_copy:
    print(" -", LOCAL_WORK_DIR / name)

Copied files to /home/nothing/shipd_problems/Email Subject Line Matching/working
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/submission.csv
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/best_params.json
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/submission_candidate_1.csv
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/submission_candidate_2.csv
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/submission_candidate_3.csv
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/best_params_candidate_1.json
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/best_params_candidate_2.json
 - /home/nothing/shipd_problems/Email Subject Line Matching/working/best_params_candidate_3.json
